# Day 3 — MLP (makemore Part 2, video 3, 1h15m)

The bigram's fatal flaw: one character of context. This video: 3 characters of context → embedding table C → hidden layer → 27-way output. (Bengio et al. 2003, the paper that started neural language modeling.)

Big NEW concepts to watch for (none of these existed in my week so far):
- **Embeddings** — characters become learned vectors, not one-hots
- **Minibatches** — training on random subsets, not the full data every step
- **Learning-rate finding** — the principled way, not vibes
- **Train/dev/test splits and overfitting** — why one loss number stops being enough

MY exercises: dataset with context window, C lookup, hidden layer forward, loss (meet `F.cross_entropy`), minibatch training loop, splits, sampler. Boilerplate: loader + maps below (built them from memory twice — they're plumbing now).

In [20]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [21]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [22]:
len(words)

32033

In [23]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [24]:
def build_dataset(block_size, restrict=True):
    X, Y = [], []

    word_locl = words[5:] if restrict == True else words

    for word in word_locl:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            Y.append(ix)
            X.append(context)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

In [25]:
BLOCK_SIZE = 3

X, Y = build_dataset(BLOCK_SIZE, True)

In [26]:
X.shape

torch.Size([228114, 3])

In [27]:
Y.shape

torch.Size([228114])

In [28]:
# we need to build look up table C, which is the embedding lookup table
EMBEDDINGS_DIMENSION = 2
HIDDEN_WIDTH = 100

g = torch.Generator().manual_seed(2147483647)
W1 = torch.randn(((EMBEDDINGS_DIMENSION * BLOCK_SIZE, HIDDEN_WIDTH)), generator=g)
B1 = torch.randn(HIDDEN_WIDTH, generator=g)
C = torch.randn((27, EMBEDDINGS_DIMENSION), generator=g)

# Layer two params
# For layer 2, there are 27 outputs.
W2 = torch.randn((100, 27), generator=g)
B2 = torch.randn((27), generator=g)

parameters = [W1, B1, W2, B2, C]

for p in parameters:
    p.requires_grad = True

emb = C[X]

In [29]:
emb.shape

torch.Size([228114, 3, 2])

In [30]:
O1 = emb.view(-1, EMBEDDINGS_DIMENSION * BLOCK_SIZE) @ W1 + B1 # -1 is like telling pytorch to autofill the correct value
O1 = O1.tanh()
O1.shape

torch.Size([228114, 100])

In [31]:
logits = O1 @ W2 + B2

In [32]:
# count = logits.exp()
# prob = count/count.sum(1, keepdim=True)
# loss = -prob[torch.arange(32), Y].log().mean()
# loss

In [33]:
# Better way to calculate loss
loss = F.cross_entropy(logits, Y)
loss

tensor(21.9226, grad_fn=<NllLossBackward0>)

In [34]:
loss.backward()

In [35]:
LEARNING_RATE = 0.1
for p in parameters:
    p.data += -LEARNING_RATE * p.grad

Now, all put together to tune the model

In [36]:
X, Y = build_dataset(BLOCK_SIZE)
HIDDEN_WIDTH = 100
EMBEDDINGS_DIMENSION = 2 

In [ ]:
LEARNING_RATE = 0.01
g = torch.Generator().manual_seed(2147483647)

for i in range(100):
    # forward pass
    # There are three inputs put into the initial neural network with two dimensions: 3 × 2 = 6.
    W1 = torch.randn((EMBEDDINGS_DIMENSION * BLOCK_SIZE, HIDDEN_WIDTH), generator=g)
    B1 = torch.randn(HIDDEN_WIDTH, generator=g)
    C = torch.randn((27, EMBEDDINGS_DIMENSION), generator=g) # 27 since there are 26 alphabets + 1 . special char

    # Second layer
    W2 = torch.randn((HIDDEN_WIDTH, 27), generator=g)
    B2 = torch.randn(27, generator=g)

    parameters = [W1, B1, W2, B2, C]

    for p in parameters:
        p.requires_grad = True

    # Embedding lookup table where you can see the corresponding embedggings for any index in X from C
    emb = C[X] # size -> [xyz, BLOCK_SIZE, EMBEDDINGS_DIMENSION]
    emb_view = emb.view(-1, EMBEDDINGS_DIMENSION * BLOCK_SIZE)

    # Output of the first layer, which is the input to the second
    O1 = emb_view @ W1 + B1
    O1 = O1.tanh()
    logit = O1 @ W2 + B2
    loss = F.cross_entropy(logit, Y)

    print("loss is ", loss.item())

    for p in parameters:
        p.grad = None

    loss.backward()

    for p in parameters:
        p.data += -LEARNING_RATE * p.grad


loss is  21.922626495361328
loss is  17.528057098388672
loss is  19.212411880493164
loss is  18.613117218017578
loss is  15.70679759979248
loss is  15.864246368408203
loss is  16.847497940063477
loss is  16.886640548706055
loss is  17.985273361206055
loss is  17.85847282409668
loss is  16.717304229736328
loss is  13.773077011108398
loss is  19.931617736816406
loss is  17.336389541625977
loss is  15.23062801361084
loss is  17.08449363708496
loss is  15.324951171875
loss is  18.044750213623047
loss is  16.602802276611328
loss is  14.001396179199219
loss is  16.81340980529785
loss is  13.827643394470215
loss is  13.261041641235352
loss is  15.337809562683105
loss is  19.010591506958008
loss is  18.427234649658203
loss is  14.96611499786377
loss is  18.427040100097656
loss is  17.9001407623291
loss is  18.688833236694336
loss is  18.007108688354492
loss is  19.321632385253906
loss is  17.220115661621094
loss is  16.98773193359375
loss is  17.103561401367188
loss is  16.814289093017578
loss

In [38]:
for i in range(100):
    # forward pass
    logit = O1 @ W2 + B2
    loss = F.cross_entropy(logit, Y)

    print("loss is ", loss)

    for p in parameters:
        p.grad = None

    loss.backward()

    for p in parameters:
        p.data += -LEARNING_RATE * p.grad

loss is  tensor(15.4308, grad_fn=<NllLossBackward0>)


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.